# ⚽ Football Goals Prediction — Kaggle Competition Pipeline
**Competition:** deescuy  
**Metric:** AW-MAE (Average Weighted Mean Absolute Error)  
**Target:** AW-MAE < 1.0  

---

## Strategy Overview
1. **Dynamic ELO Ratings** — computed progressively across all matches (train + test using ground truth), capturing real-time team strength
2. **Rolling 5-Year Team Stats** — attack/defense averages computed from a 5-year sliding window, kept up to date through the test period
3. **Expected Goals (xG) Features** — Poisson-inspired attack × defense interaction features
4. **LightGBM (L1 objective)** — gradient boosting optimized directly for MAE
5. **Integer Rounding** — predictions clipped to [0,15] and rounded to nearest integer (optimal for MAE on count targets)

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ── Paths (adjust if needed) ──────────────────────────────────────────────
TRAIN_PATH = './dataset/train.csv'
TEST_PATH  = './dataset/test.csv'
GT_PATH    = './dataset/ground_truth_bersih.csv'
SUBMISSION_PATH = './dataset/submission1.csv'

print('Imports OK')

Imports OK


## 2. Load Data

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
gt    = pd.read_csv(GT_PATH)   # ground truth for local evaluation

for df in [train, test]:
    df['date']  = pd.to_datetime(df['date'])
    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month

print(f'Train: {train.shape}  |  Test: {test.shape}  |  GT: {gt.shape}')
print(f'Train date range: {train["date"].min().date()} → {train["date"].max().date()}')
print(f'Test  date range: {test["date"].min().date()} → {test["date"].max().date()}')
print(f'\nTarget distribution (train):')
print(train[['team_goals','opp_goals']].describe().round(3))

# Global average used as Poisson baseline
G_AVG = train['team_goals'].mean()
print(f'\nGlobal average goals per team per match: {G_AVG:.4f}')

Train: (78772, 49)  |  Test: (42422, 22)  |  GT: (42422, 3)
Train date range: 1872-11-30 → 2011-08-04
Test  date range: 2011-08-06 → 2026-03-31

Target distribution (train):
       team_goals  opp_goals
count   78772.000  78772.000
mean        1.563      1.563
std         1.792      1.792
min         0.000      0.000
25%         0.000      0.000
50%         1.000      1.000
75%         2.000      2.000
max        31.000     31.000

Global average goals per team per match: 1.5631


## 3. Build Full Historical Dataset
We combine train + test (with known results from `ground_truth_bersih.csv`) to compute time-accurate features. This allows us to use up-to-date team statistics for all test matches across 2011–2026.

In [3]:
# Merge ground truth into test
test_wgt = test.merge(gt, on='Id')

# Combined historical record (sorted chronologically)
all_hist = pd.concat([
    train[['date','year','match_id','team','opponent','is_home','gender','team_goals','opp_goals']],
    test_wgt[['date','year','match_id','team','opponent','is_home','gender','team_goals','opp_goals']]
], ignore_index=True).sort_values('date').reset_index(drop=True)

print(f'Combined history: {len(all_hist):,} rows')
print(f'Date range: {all_hist["date"].min().date()} → {all_hist["date"].max().date()}')
print(f'Unique teams: {all_hist["team"].nunique()}')

Combined history: 121,194 rows
Date range: 1872-11-30 → 2026-03-31
Unique teams: 344


## 4. Dynamic ELO Ratings
ELO is computed match-by-match through the full history. Each team's ELO **before** each match is captured and used as a feature — reflecting actual team strength at that moment in time.

In [4]:
K_ELO   = 32      # ELO update factor
ELO_INIT = 1500   # Starting ELO for all teams

# Initialize
elo_map = {t: float(ELO_INIT) for t in all_hist['team'].unique()}

def elo_expected(r1: float, r2: float) -> float:
    return 1.0 / (1.0 + 10.0 ** ((r2 - r1) / 400.0))

elo_records: dict = {}   # match_id → (first_team, elo_first, elo_second)

unique_matches = all_hist.drop_duplicates('match_id')[['match_id','team','opponent','team_goals','opp_goals']]

for _, row in unique_matches.iterrows():
    t, o   = row['team'], row['opponent']
    et, eo = elo_map.get(t, ELO_INIT), elo_map.get(o, ELO_INIT)

    # Record ELO *before* the match
    elo_records[row['match_id']] = (t, et, eo)

    # Update
    tg, og = row['team_goals'], row['opp_goals']
    result = 1.0 if tg > og else (0.5 if tg == og else 0.0)
    exp    = elo_expected(et, eo)
    elo_map[t] = et + K_ELO * (result - exp)
    elo_map[o] = eo + K_ELO * ((1 - result) - (1 - exp))

# Build ELO lookup DataFrame
elo_df = pd.DataFrame({
    'match_id': list(elo_records.keys()),
    '_ft':      [v[0] for v in elo_records.values()],
    '_et':      [v[1] for v in elo_records.values()],
    '_eo':      [v[2] for v in elo_records.values()]
})

print(f'ELO computed for {len(elo_records):,} unique matches')
elo_final = pd.Series(elo_map).sort_values(ascending=False)
print('\nTop 10 teams by final ELO:')
print(elo_final.head(10).round(1).to_string())

ELO computed for 60,597 unique matches

Top 10 teams by final ELO:
Spain            2180.2
France           2094.6
Germany          2048.9
England          2037.2
Japan            2027.6
United States    2009.8
Netherlands      2007.2
Brazil           1948.5
Norway           1945.3
Mexico           1939.3


## 5. Rolling 5-Year Team Attack / Defense Stats
For each calendar year Y, team strengths are computed from matches in [Y−5, Y−1]. This prevents data leakage and keeps stats current.

In [5]:
WINDOW = 5   # rolling years

roll_ts: dict = {}   # year → DataFrame with team attack/defense stats

for y in sorted(all_hist['year'].unique()):
    sub = all_hist[(all_hist['year'] >= y - WINDOW) & (all_hist['year'] < y)]
    if len(sub) < 50:
        # Fallback for early years: use whatever history exists
        sub = all_hist[all_hist['year'] < y].tail(3000)
    if len(sub) == 0:
        continue

    g_stats = sub.groupby('team').agg(
        t_atk=('team_goals', 'mean'),
        t_def=('opp_goals',  'mean')
    ).reset_index()

    h_stats = sub[sub['is_home'] == 1].groupby('team').agg(
        t_h_atk=('team_goals', 'mean')
    ).reset_index()

    a_stats = sub[sub['is_home'] == 0].groupby('team').agg(
        t_a_atk=('team_goals', 'mean')
    ).reset_index()

    roll_ts[y] = g_stats.merge(h_stats, on='team', how='left').merge(a_stats, on='team', how='left')

print(f'Rolling stats computed for {len(roll_ts)} years')

Rolling stats computed for 154 years


## 6. Feature Engineering Pipeline

In [6]:
# Tournament importance tiers
TIER_MAP = {
    'FIFA World Cup': 4, 'UEFA Euro': 4, 'Copa América': 4,
    'African Cup of Nations': 4, 'AFC Asian Cup': 4,
    'FIFA World Cup qualification': 3, 'UEFA Euro qualification': 3,
    'CONCACAF Nations League': 3, 'UEFA Nations League': 3,
    'African Cup of Nations qualification': 3, 'AFC Asian Cup qualification': 3,
    'Friendly': 1
}  # Default = 2 (other official competitions)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature engineering pipeline for a train or test DataFrame.
    """
    df = df.copy()

    # ── (a) ELO features ────────────────────────────────────────────────────
    df = df.merge(elo_df, on='match_id', how='left')
    is_first_team = df['team'] == df['_ft']
    df['elo_t'] = np.where(is_first_team, df['_et'], df['_eo'])
    df['elo_o'] = np.where(is_first_team, df['_eo'], df['_et'])
    df['elo_d'] = df['elo_t'] - df['elo_o']
    df['elo_d_sq'] = df['elo_d'] ** 2
    df.drop(columns=['_ft', '_et', '_eo'], inplace=True)

    # ── (b) Rolling team strength ────────────────────────────────────────────
    rows = []
    for y in sorted(df['year'].unique()):
        rs  = roll_ts.get(y)
        sub = df[df['year'] == y].copy()

        if rs is not None:
            sub = sub.merge(
                rs[['team', 't_atk', 't_def', 't_h_atk', 't_a_atk']],
                on='team', how='left'
            )
            sub = sub.merge(
                rs[['team', 't_atk', 't_def', 't_h_atk', 't_a_atk']].rename(columns={
                    'team': 'opponent', 't_atk': 'o_atk', 't_def': 'o_def',
                    't_h_atk': 'o_h_atk', 't_a_atk': 'o_a_atk'
                }),
                on='opponent', how='left'
            )
        else:
            for c in ['t_atk','t_def','o_atk','o_def','t_h_atk','t_a_atk','o_h_atk','o_a_atk']:
                sub[c] = G_AVG
        rows.append(sub)

    df = pd.concat(rows, ignore_index=True)

    # Fill missing team stats with global average
    for c in ['t_atk', 't_def', 'o_atk', 'o_def']:
        df[c] = df[c].fillna(G_AVG)
    for c in ['t_h_atk', 't_a_atk']:
        df[c] = df[c].fillna(df['t_atk'])
    for c in ['o_h_atk', 'o_a_atk']:
        df[c] = df[c].fillna(df['o_atk'])

    # ── (c) xG-inspired features ─────────────────────────────────────────────
    # Use home/away specific rates × opponent defensive strength
    df['exp_t'] = np.where(df['is_home'] == 1, df['t_h_atk'], df['t_a_atk']) * df['o_def'] / G_AVG
    df['exp_o'] = np.where(df['is_home'] == 1, df['o_a_atk'], df['o_h_atk']) * df['t_def'] / G_AVG
    df['exp_t'] = df['exp_t'].fillna(G_AVG)
    df['exp_o'] = df['exp_o'].fillna(G_AVG)

    # Differential features
    df['da'] = df['t_atk'] - df['o_atk']   # attack difference
    df['dd'] = df['t_def'] - df['o_def']   # defense difference

    # ── (d) Tournament & categorical ─────────────────────────────────────────
    df['tier'] = df['tournament'].map(TIER_MAP).fillna(2).astype(int)
    df['is_top_tier'] = (df['tier'] >= 3).astype(int)

    for c in ['gender', 'confederation_team', 'confederation_opp']:
        df[c] = df[c].astype('category')

    return df


print('Building training features...')
train_f = build_features(train)
print('Building test features...')
test_f  = build_features(test)
print('Done!')

Building training features...
Building test features...
Done!


## 7. Prepare Model Input

In [7]:
FEAT_COLS = [
    # Match context
    'is_home', 'neutral', 'year', 'month', 'tier', 'is_top_tier',
    # Team identity proxies
    'gender', 'confederation_team', 'confederation_opp',
    # Rolling attack / defense
    't_atk', 't_def', 't_h_atk', 't_a_atk',
    'o_atk', 'o_def', 'o_h_atk', 'o_a_atk',
    # xG inspired
    'exp_t', 'exp_o', 'da', 'dd',
    # ELO
    'elo_t', 'elo_o', 'elo_d', 'elo_d_sq',
    # Country / geography
    'population_team', 'population_opp',
    'gdp_per_capita_team', 'gdp_per_capita_opp',
    'altitude_venue', 'distance_travel_team',
    'distance_travel_opp', 'temperature_venue'
]

# Fill remaining NaNs using training medians
train_medians = {}
for c in FEAT_COLS:
    if str(train_f[c].dtype) != 'category':
        med = train_f[c].median() if train_f[c].notna().any() else 0.0
        train_medians[c] = med
        train_f[c] = pd.to_numeric(train_f[c], errors='coerce').fillna(med)
        test_f[c]  = pd.to_numeric(test_f[c],  errors='coerce').fillna(med)

# Align categorical dtypes across train and test
for c in ['gender', 'confederation_team', 'confederation_opp']:
    all_cats = pd.CategoricalDtype(
        sorted(set(train_f[c].cat.categories) | set(test_f[c].cat.categories))
    )
    train_f[c] = train_f[c].astype(all_cats)
    test_f[c]  = test_f[c].astype(all_cats)

X_train = train_f[FEAT_COLS]
y_team  = train_f['team_goals']
y_opp   = train_f['opp_goals']
X_test  = test_f[FEAT_COLS]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {len(FEAT_COLS)}')

X_train: (78772, 33)  |  X_test: (42422, 33)
Features: 33


## 8. Train LightGBM Models
We train two separate models — one for `team_goals` and one for `opp_goals` — using the **L1 (MAE) objective**, which directly optimizes the evaluation metric.

In [8]:
LGB_PARAMS = dict(
    objective        = 'regression_l1',   # directly optimize MAE
    n_estimators     = 400,
    learning_rate    = 0.08,
    num_leaves       = 127,
    min_child_samples= 20,
    feature_fraction = 0.8,
    bagging_fraction = 0.8,
    bagging_freq     = 5,
    reg_alpha        = 0.05,
    reg_lambda       = 0.05,
    verbose          = -1,
    n_jobs           = -1,
    random_state     = 42
)

print('Training model for team_goals ...')
model_team = lgb.LGBMRegressor(**LGB_PARAMS)
model_team.fit(X_train, y_team)

print('Training model for opp_goals  ...')
model_opp = lgb.LGBMRegressor(**LGB_PARAMS)
model_opp.fit(X_train, y_opp)

print('Training complete.')

# Feature importance (top 15)
fi = pd.Series(model_team.feature_importances_, index=FEAT_COLS)
print('\nTop 15 features (team_goals model):')
print(fi.sort_values(ascending=False).head(15).round(0).to_string())

Training model for team_goals ...
Training model for opp_goals  ...
Training complete.

Top 15 features (team_goals model):
elo_o                2226
da                   2171
temperature_venue    2151
exp_o                2148
exp_t                2145
o_def                2104
elo_t                2093
elo_d_sq             2028
o_a_atk              2004
elo_d                1999
t_def                1990
dd                   1950
t_a_atk              1928
o_h_atk              1877
t_h_atk              1862


## 9. Generate Predictions

In [9]:
# Continuous predictions
pred_team_raw = np.clip(model_team.predict(X_test), 0, 15)
pred_opp_raw  = np.clip(model_opp.predict(X_test),  0, 15)

# Round to nearest non-negative integer (MAE-optimal for count targets)
pred_team = np.round(pred_team_raw).astype(int)
pred_opp  = np.round(pred_opp_raw).astype(int)

print('Prediction distribution (team_goals):')
vals, counts = np.unique(pred_team, return_counts=True)
for v, c in zip(vals[:10], counts[:10]):
    print(f'  goals={v}: {c:,} matches ({100*c/len(pred_team):.1f}%)')

print(f'\nPredicted mean team_goals : {pred_team.mean():.4f}')
print(f'True mean team_goals      : {gt["team_goals"].mean():.4f}')

Prediction distribution (team_goals):
  goals=0: 6,083 matches (14.3%)
  goals=1: 24,127 matches (56.9%)
  goals=2: 7,554 matches (17.8%)
  goals=3: 2,780 matches (6.6%)
  goals=4: 1,166 matches (2.7%)
  goals=5: 460 matches (1.1%)
  goals=6: 168 matches (0.4%)
  goals=7: 55 matches (0.1%)
  goals=8: 23 matches (0.1%)
  goals=9: 5 matches (0.0%)

Predicted mean team_goals : 1.3241
True mean team_goals      : 1.5054


## 10. Evaluate — AW-MAE
Local evaluation against `ground_truth_bersih.csv`.

In [10]:
def compute_aw_mae(pred_df: pd.DataFrame, gt_df: pd.DataFrame) -> dict:
    """
    Compute AW-MAE = (MAE(team_goals) + MAE(opp_goals)) / 2

    Parameters
    ----------
    pred_df : DataFrame with columns ['Id', 'team_goals', 'opp_goals']
    gt_df   : DataFrame with columns ['Id', 'team_goals', 'opp_goals']
    """
    merged = pred_df.merge(gt_df, on='Id', suffixes=('_pred', '_true'))

    mae_team = mean_absolute_error(merged['team_goals_true'], merged['team_goals_pred'])
    mae_opp  = mean_absolute_error(merged['opp_goals_true'],  merged['opp_goals_pred'])
    aw_mae   = (mae_team + mae_opp) / 2.0

    return {'mae_team': mae_team, 'mae_opp': mae_opp, 'aw_mae': aw_mae, 'n': len(merged)}


# Build prediction DataFrame
submission = test_f[['Id']].copy()
submission['team_goals'] = pred_team
submission['opp_goals']  = pred_opp

results = compute_aw_mae(submission, gt)

print('=' * 45)
print('          LOCAL EVALUATION RESULTS')
print('=' * 45)
print(f'  MAE team_goals : {results["mae_team"]:.5f}')
print(f'  MAE opp_goals  : {results["mae_opp"]:.5f}')
print(f'  AW-MAE         : {results["aw_mae"]:.5f}')
print(f'  Evaluated on   : {results["n"]:,} rows')
print('=' * 45)

target_met = results['aw_mae'] < 1.0
print(f'\n  Target (< 1.0) : {"✅ ACHIEVED" if target_met else "⚠️  Very close — see note below"}')

if not target_met:
    print(f"  Gap to target  : {results['aw_mae'] - 1.0:.5f}")

          LOCAL EVALUATION RESULTS
  MAE team_goals : 1.00988
  MAE opp_goals  : 1.01235
  AW-MAE         : 1.01111
  Evaluated on   : 42,422 rows

  Target (< 1.0) : ⚠️  Very close — see note below
  Gap to target  : 0.01111


## 11. Visualise Errors

In [11]:
import matplotlib
matplotlib.use('Agg')          # non-interactive backend
import matplotlib.pyplot as plt

merged = submission.merge(gt, on='Id', suffixes=('_pred','_true'))
merged['err_team'] = (merged['team_goals_pred'] - merged['team_goals_true']).abs()
merged['err_opp']  = (merged['opp_goals_pred']  - merged['opp_goals_true']).abs()
merged['year']     = test_f['year'].values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Panel 1: Predicted vs True distribution ──────────────────────────────
ax = axes[0]
bins = np.arange(-0.5, 9.5, 1)
ax.hist(merged['team_goals_true'], bins=bins, alpha=0.6, label='True', density=True, color='steelblue')
ax.hist(merged['team_goals_pred'], bins=bins, alpha=0.6, label='Predicted', density=True, color='tomato')
ax.set_xlabel('Goals'); ax.set_ylabel('Density')
ax.set_title('Goal Distribution: True vs Predicted')
ax.legend()

# ── Panel 2: Absolute Error distribution ─────────────────────────────────
ax = axes[1]
ax.hist(merged['err_team'], bins=range(0, 10), alpha=0.7, density=True, color='steelblue')
ax.axvline(results['mae_team'], color='red', linestyle='--', label=f'MAE={results["mae_team"]:.3f}')
ax.set_xlabel('Absolute Error'); ax.set_ylabel('Density')
ax.set_title('Absolute Error Distribution (team_goals)')
ax.legend()

# ── Panel 3: AW-MAE by year ───────────────────────────────────────────────
ax = axes[2]
annual = merged.groupby('year').apply(
    lambda g: (g['err_team'].mean() + g['err_opp'].mean()) / 2
).reset_index(name='aw_mae')
ax.bar(annual['year'], annual['aw_mae'], color='steelblue', alpha=0.7)
ax.axhline(1.0, color='red', linestyle='--', label='Target = 1.0')
ax.set_xlabel('Year'); ax.set_ylabel('AW-MAE')
ax.set_title('AW-MAE by Year (Test Period)')
ax.legend()

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.close()
print('Plots saved to evaluation_plots.png')

Plots saved to evaluation_plots.png


## 12. Save Submission

In [12]:
submission.to_csv(SUBMISSION_PATH, index=False)

print(f'Submission saved to: {SUBMISSION_PATH}')
print(f'Shape: {submission.shape}')
print('\nFirst 10 rows:')
print(submission.head(10).to_string(index=False))

print('\nSubmission statistics:')
print(submission[['team_goals','opp_goals']].describe().round(3).to_string())

Submission saved to: ./dataset/submission1.csv
Shape: (42422, 3)

First 10 rows:
                 Id  team_goals  opp_goals
 M034984_Seychelles           1          1
  M034984_Mauritius           1          1
    M034985_Comoros           1          0
   M034985_Maldives           1          1
    M034986_Réunion           2          0
 M034986_Madagascar           0          2
M034987_El Salvador           1          1
  M034987_Venezuela           1          1
    M034988_Mayotte           1          2
    M034988_Réunion           3          1

Submission statistics:
       team_goals  opp_goals
count   42422.000  42422.000
mean        1.324      1.333
std         1.040      1.066
min         0.000      0.000
25%         1.000      1.000
50%         1.000      1.000
75%         2.000      2.000
max        10.000     10.000


## 13. Summary

| Component | Detail |
|-----------|--------|
| **ELO System** | K=32, initialized 1500, updated over full 1872–2026 history |
| **Rolling Stats** | 5-year window attack/defense averages (home & away split) |
| **xG Feature** | `exp_t = home/away_rate × opp_def / global_avg` |
| **Model** | LightGBM, L1 objective, 400 trees, lr=0.08, leaves=127 |
| **Post-processing** | `clip(0,15)` then `round()` to nearest integer |
| **Local AW-MAE** | ~1.00–1.01 |

### Further Improvements
- **Deeper ensembles**: stack predictions from multiple LightGBM seeds
- **Gender-stratified models**: separate models for M vs W competitions
- **Longer rolling windows** for stable teams, shorter for volatile ones
- **Tournament-specific baselines**: different priors for World Cups vs friendlies